# Construct End-User Parquet Files

In [11]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import uproot
import glob
import awkward as ak
import itertools
import yaml
import os
import sys
from tqdm import tqdm
from pathlib import Path
import atlasify as atl
from typing import List
import h5py
atl.ATLAS = "ColliderML"

sys.path.append("../")
from utils import load_root_file, load_hepmc_event
from edm4hep_utils import build_calo_df, build_particle_df, build_tracker_df, load_edm4hep_file
from clustering_metrics import evaluate_clustering, plot_clustering_metrics

from edm4hep_utils import pixel_readouts, strip_readouts
all_tracker_readouts = pixel_readouts + strip_readouts

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Roadmap

1. Load in edm4hep file
2. Load in ambi tracks csv 
3. Load in particles root
4. Load in measurements root
5. Load in trackstates ambi root
6. Load in tracksummary ambi root
7. Create track parquet object
8. Track object needs:
    - hits in the track (hit ID)
    - track fit parameters
    - track quality parameters
    - track states at IP, end-of-track, first hit, last hit

# Loading

## 1. Load in edm4hep file

In [6]:
edm4hep_file = "/eos/home-d/dmurnane/www/ColliderML/simulation/gg2ttbar/v1/runs/0/edm4hep.root"
# edm4hep_file = "/eos/home-d/dmurnane/www/ColliderML/simulation/chichi/v1/runs/0/edm4hep.root"
event = load_edm4hep_file(edm4hep_file, event_num=0)

In [7]:
event.keys()

dict_keys(['tracker_df', 'calo_hits_df', 'calo_contrib_df', 'particles_df', 'parents_df', 'daughters_df'])

In [8]:
tracker_df = event["tracker_df"]
tracker_df.columns

Index(['cellID', 'EDep', 'time', 'pathLength', 'quality', 'x', 'y', 'z', 'px',
       'py', 'pz', 'particle_id', 'r', 'R', 'phi', 'theta', 'eta', 'pt',
       'detector'],
      dtype='object')

In [9]:
parents_df = event["parents_df"]
parents_df.columns

particles_df = event["particles_df"]
# Create a column from the index
particles_df["particle_id"] = particles_df.index
particles_df.columns

Index(['PDG', 'generatorStatus', 'simulatorStatus', 'charge', 'time', 'mass',
       'vx', 'vy', 'vz', 'px', 'py', 'pz', 'endpoint_x', 'endpoint_y',
       'endpoint_z', 'parents_begin', 'parents_end', 'daughters_begin',
       'daughters_end', 'pt', 'p', 'eta', 'phi', 'particle_id'],
      dtype='object')

# Hits Objects

In [12]:
def get_run_paths(base_dir: str) -> List[str]:
    """
    Get all run directories in the dataset, properly sorted.
    """
    # Get all run directories and sort numerically by run number
    run_dirs = glob.glob(f"{base_dir}/runs/*/")
    return sorted(run_dirs, key=lambda x: int(x.rstrip('/').split('/')[-1]))

def process_event_for_hits(
    event_id: int,
    tracker_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Process hit data for a single event using EDM4HEP tracker hits.
    
    Args:
        event_id: Event number
        tracker_df: DataFrame containing EDM4HEP tracker hits
        
    Returns:
        DataFrame containing hit data for this event
    """
    # Select relevant columns
    hit_columns = [
        "cellID",
        "EDep",
        "time",
        "pathLength",
        "quality",
        "x",
        "y",
        "z",
        "px",
        "py",
        "pz",
        "particle_id",
        "detector",
    ]
    
    event_hits = tracker_df[hit_columns].copy()
    
    # Add event_id
    event_hits["event_id"] = event_id
    
    return event_hits

def build_hdf5_hits(
    df: pd.DataFrame,
    output_file: str
) -> None:
    """
    Build HDF5 file with event/hit hierarchy.
    
    Structure:
    /events/
        /event_0/
            /hits    # Dataset containing hit properties
        /event_1/
            ...
    """
    with h5py.File(output_file, 'a') as f:
        # Create events group if it doesn't exist
        if 'events' not in f:
            events_group = f.create_group('events')
        else:
            events_group = f['events']
            
        # Group DataFrame by event_id
        for event_id, event_df in df.groupby('event_id'):
            # Create event group
            event_group = events_group.create_group(f'event_{event_id}')
            
            # Store hit data
            event_group.create_dataset(
                'hits',
                data=event_df.drop(columns=['event_id']).to_records(index=False),
                compression="gzip",
                compression_opts=9
            )

def process_full_dataset_for_hits(
    base_dir: str,
    output_base_dir: str,
    chunk_size: int = 1000,
    run_size: int = 10,
    dataset_name: str = "pileup-10/ttbar/v1/reco/hits/"
) -> None:
    """
    Process entire dataset in chunks.
    """
    # Get properly sorted run directories
    run_dirs = get_run_paths(base_dir)
    num_runs = len(run_dirs)
    num_events = num_runs * run_size
    
    # Calculate runs per chunk
    runs_per_chunk = chunk_size // run_size
    
    print(f"Processing {num_runs} runs with {num_events} total events")
    print(f"Processing {runs_per_chunk} runs per chunk to get ~{chunk_size} events per file")
    
    output_dir = f"{output_base_dir}/{dataset_name}"
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    dataset_name = dataset_name.replace("/", ".")
    
    # Process chunks of runs
    for start_run in tqdm(range(0, num_runs, runs_per_chunk), desc="Processing chunks"):
        all_events_data = []
        
        # Process each run in the chunk
        for run_idx in range(start_run, min(start_run + runs_per_chunk, len(run_dirs))):
            run_dir = run_dirs[run_idx]
            print(f"Processing run {run_idx} at {run_dir}")
            
            # Load EDM4HEP file
            edm4hep_file = f"{run_dir}/edm4hep.root"
            event = load_edm4hep_file(edm4hep_file, event_num=0)
            
            # Process event
            event_df = process_event_for_hits(
                run_idx * run_size,
                event["tracker_df"]
            )
            all_events_data.append(event_df)
            
        if all_events_data:
            combined_df = pd.concat(all_events_data, ignore_index=True)
            
            # Calculate event range for filename
            start_event = start_run * run_size
            end_event = min((start_run + runs_per_chunk) * run_size - 1, 
                           len(run_dirs) * run_size - 1)
            
            output_file = f"{output_dir}/{dataset_name}.events{start_event}-{end_event}.h5"
            build_hdf5_hits(combined_df, output_file)
            print(f"Saved {output_file}")

In [13]:
def read_event_hits(
    input_file: str,
    event_id: int
) -> pd.DataFrame:
    """
    Read hit data for a specific event.
    """
    with h5py.File(input_file, 'r') as f:
        event_group = f[f'events/event_{event_id}']
        
        # Read hit data
        hits = event_group['hits'][:]
        
        # Convert to DataFrame
        df = pd.DataFrame(hits)
        df['event_id'] = event_id
        return df

def read_events_hits(
    base_dir: str,
    event_ids: List[int],
    dataset_name: str = "mu10.ttbar"
) -> pd.DataFrame:
    """
    Read specific events from HDF5 files.
    """
    events_data = []
    chunk_size = 1000  # Must match the chunk size used in writing
    
    for event_id in event_ids:
        # Calculate which file contains this event
        chunk_num = event_id // chunk_size
        start_event = chunk_num * chunk_size
        
        # Find the file
        pattern = f"{base_dir}/{dataset_name}.hits.events{start_event}-*.h5"
        matching_files = glob.glob(pattern)
        
        if not matching_files:
            print(f"Could not find file containing event {event_id}")
            continue
            
        filename = matching_files[0]
        
        try:
            event_df = read_event_hits(filename, event_id)
            events_data.append(event_df)
        except KeyError as e:
            print(f"Could not read event {event_id}: {e}")
            continue
    
    return pd.concat(events_data, ignore_index=True) if events_data else pd.DataFrame()

In [14]:
# Process the full dataset
base_dir = "/eos/home-d/dmurnane/www/ColliderML/simulation/gg2ttbar/v1"
output_base_dir = "/eos/home-d/dmurnane/ColliderML/staging/"
dataset_name = "pileup-10/ttbar/v1/truth/hits"


In [15]:
process_full_dataset_for_hits(base_dir, output_base_dir, dataset_name=dataset_name)

Processing 128 runs with 1280 total events
Processing 100 runs per chunk to get ~1000 events per file


Processing chunks:   0%|          | 0/2 [00:00<?, ?it/s]

Processing run 0 at /eos/home-d/dmurnane/www/ColliderML/simulation/gg2ttbar/v1/runs/0/
Processing run 1 at /eos/home-d/dmurnane/www/ColliderML/simulation/gg2ttbar/v1/runs/1/
Processing run 2 at /eos/home-d/dmurnane/www/ColliderML/simulation/gg2ttbar/v1/runs/2/


Processing chunks:   0%|          | 0/2 [00:24<?, ?it/s]


KeyboardInterrupt: 

In [ ]:

# Read back specific events
events_df = read_events_hits(
    "/eos/home-d/dmurnane/ColliderML/staging/pileup-10/ttbar/v1/hits",
    event_ids=[0, 42, 999, 1010],
    dataset_name="pileup-10.ttbar"
)